In [6]:
import torch
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry
from torch.optim import Adam
import torch.nn as nn
from glob import glob
import random

# ==================== CONFIGURATION ====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_image_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images'
train_mask_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_1cmasks'
test_image_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_images'
output_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output'
point_masks_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_point_1cmasks'
sam_checkpoint = "/home/iiitdmk-param/Desktop/sam_vit_b.pth"
model_type = "vit_b"

# Create output directories
os.makedirs(output_dir, exist_ok=True)
os.makedirs(point_masks_dir, exist_ok=True)

# ==================== CLASS DEFINITIONS ====================
CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]},
    18: {"name": "point_mask", "rgb": [100, 100, 100]}
}

IGNORE_CLASS_RGB = [100, 100, 100]
IGNORE_CLASS_ID = 255

NUM_CLASSES = len(CLASS_INFO)
print(f"Number of classes: {NUM_CLASSES}")

# ==================== FIXED MULTI-CLASS SAM WRAPPER ====================
class MultiClassSAMWrapper(nn.Module):
    def __init__(self, sam_model, num_classes):
        super().__init__()
        self.sam = sam_model
        self.num_classes = num_classes
        
        # Get the correct input channels for the multi-class head
        # SAM's mask decoder outputs masks with a specific number of channels
        # For vit_b, it's typically 1 channel for masks
        self.mask_decoder_output_channels = 1  # SAM outputs single channel masks
        
        # Add custom multi-class head that matches SAM's output
        self.multi_class_head = nn.Sequential(
            nn.Conv2d(self.mask_decoder_output_channels, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, 1)
        )
        
        # Freeze SAM parameters, only train custom head
        for param in self.sam.parameters():
            param.requires_grad = False
    
    def forward_single(self, x, point_coords=None, point_labels=None):
        """Process single image with point prompts"""
        # Get image embeddings from SAM
        with torch.no_grad():
            image_embeddings = self.sam.image_encoder(x.unsqueeze(0))
        
        # If point prompts are provided, use mask decoder
        if point_coords is not None and point_labels is not None:
            # Prepare points in the format expected by SAM
            # SAM expects: points: (1, N, 2), labels: (1, N)
            point_coords = point_coords.unsqueeze(0)  # [N, 2] -> [1, N, 2]
            point_labels = point_labels.unsqueeze(0)  # [N] -> [1, N]
            
            # Get prompt embeddings
            sparse_embeddings, dense_embeddings = self.sam.prompt_encoder(
                points=(point_coords, point_labels),
                boxes=None,
                masks=None,
            )
            
            # Use mask decoder with prompts
            low_res_masks, iou_predictions = self.sam.mask_decoder(
                image_embeddings=image_embeddings,
                image_pe=self.sam.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=False,
            )
            
            # Pass through multi-class head
            logits = self.multi_class_head(low_res_masks)
            return logits.squeeze(0)  # Remove batch dimension
        else:
            # No prompts - use a simpler approach
            # We'll create a dummy mask and use that
            b, c, h, w = image_embeddings.shape
            dummy_mask = torch.zeros((b, self.mask_decoder_output_channels, h, w), 
                                   device=image_embeddings.device)
            logits = self.multi_class_head(dummy_mask)
            return logits.squeeze(0)
    
    def forward(self, x, point_coords_list=None, point_labels_list=None):
        """Process batch of images"""
        batch_size = x.shape[0]
        all_logits = []
        
        for i in range(batch_size):
            if point_coords_list is not None and point_labels_list is not None:
                logits = self.forward_single(
                    x[i], 
                    point_coords_list[i], 
                    point_labels_list[i]
                )
            else:
                logits = self.forward_single(x[i])
            all_logits.append(logits.unsqueeze(0))
        
        return torch.cat(all_logits, dim=0)

# ==================== MASK PROCESSING FUNCTIONS ====================
def process_mask_to_ignore_class(mask, ignore_rgb=IGNORE_CLASS_RGB):
    if len(mask.shape) == 3:
        h, w, c = mask.shape
        processed_mask = np.zeros((h, w), dtype=np.uint8)
        ignore_pixels = np.all(mask == ignore_rgb, axis=-1)
        processed_mask[ignore_pixels] = IGNORE_CLASS_ID
        for class_id, info in CLASS_INFO.items():
            class_rgb = info["rgb"]
            class_pixels = np.all(mask == class_rgb, axis=-1)
            processed_mask[class_pixels] = class_id
    else:
        processed_mask = mask.copy()
    return processed_mask

# ==================== POINT MASK ANALYSIS FUNCTIONS ====================
def analyze_point_masks(point_masks_dir):
    point_mask_files = sorted([f for f in os.listdir(point_masks_dir) if f.endswith('.png')])
    print("=== POINT MASK ANALYSIS ===")
    for i, mask_file in enumerate(point_mask_files[:5]):
        mask_path = os.path.join(point_masks_dir, mask_file)
        point_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if point_mask is not None:
            unique_vals = np.unique(point_mask)
            print(f"{mask_file}: Shape: {point_mask.shape}, Unique values: {unique_vals}")
            has_ignore = 18 in unique_vals
            print(f"  Contains ignore class (18): {has_ignore}")
    print(f"Total point masks found: {len(point_mask_files)}")

def extract_points_alternative(point_mask):
    point_coords = []
    point_labels = []
    h, w = point_mask.shape
    foreground_mask = (point_mask > 0) & (point_mask != 18)
    foreground_coords = np.argwhere(foreground_mask)
    for coord in foreground_coords:
        y, x = coord
        point_coords.append([x, y])
        point_labels.append(1)
    if len(point_coords) == 0:
        valid_mask = point_mask != 18
        valid_coords = np.argwhere(valid_mask)
        if len(valid_coords) > 0:
            num_points = min(5, len(valid_coords))
            sampled_indices = np.random.choice(len(valid_coords), num_points, replace=False)
            for idx in sampled_indices:
                y, x = valid_coords[idx]
                point_coords.append([x, y])
                class_val = point_mask[y, x]
                point_labels.append(1 if class_val > 0 else 0)
    return point_coords, point_labels

# ==================== DATASET AND TRANSFORMS ====================
class SimpleTransform:
    def __call__(self, image, mask):
        image = cv2.resize(image, (1024, 1024))
        mask = cv2.resize(mask, (1024, 1024), interpolation=cv2.INTER_NEAREST)
        mask = process_mask_to_ignore_class(mask)
        image = image.astype(np.float32) / 255.0
        image = (image - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).long()
        return image, mask

class MultiClassSAMDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_name = self.image_files[idx].split('.')[0] + '.png'
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Could not load image: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path)
        if mask is None:
            mask = np.zeros((image.shape[0], image.shape[1], 3), dtype=np.uint8)
        if len(mask.shape) == 2:
            mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        
        if self.transform:
            image, mask = self.transform(image, mask)
        
        return {'image': image, 'mask': mask}

# ==================== SIMPLIFIED TRAINING FUNCTION ====================
def train_multi_class_sam(model, train_loader, val_loader, num_epochs=100, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_CLASS_ID)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            optimizer.zero_grad()
            
            # Generate point prompts for each image in batch
            batch_point_coords = []
            batch_point_labels = []
            
            for i in range(images.size(0)):
                mask_np = masks[i].cpu().numpy()
                point_coords, point_labels = generate_point_prompts(mask_np, num_points=5)
                
                # Convert to tensor and normalize coordinates
                point_coords_tensor = torch.from_numpy(point_coords).float().to(device)
                point_labels_tensor = torch.from_numpy(point_labels).float().to(device)
                
                # Normalize coordinates to [0,1] range
                point_coords_tensor = point_coords_tensor / torch.tensor([1024, 1024]).to(device)
                
                batch_point_coords.append(point_coords_tensor)
                batch_point_labels.append(point_labels_tensor)
            
            # Forward pass with point prompts
            logits = model(images, batch_point_coords, batch_point_labels)
            
            # Resize logits to match mask size if needed
            if logits.shape[2:] != masks.shape[1:]:
                logits = torch.nn.functional.interpolate(
                    logits, 
                    size=masks.shape[1:], 
                    mode='bilinear', 
                    align_corners=False
                )
            
            loss = criterion(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            
            if batch_idx % 5 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(device)
                masks = batch['mask'].to(device)
                
                batch_point_coords = []
                batch_point_labels = []
                for i in range(images.size(0)):
                    mask_np = masks[i].cpu().numpy()
                    point_coords, point_labels = generate_point_prompts(mask_np, num_points=5)
                    point_coords_tensor = torch.from_numpy(point_coords).float().to(device)
                    point_labels_tensor = torch.from_numpy(point_labels).float().to(device)
                    point_coords_tensor = point_coords_tensor / torch.tensor([1024, 1024]).to(device)
                    batch_point_coords.append(point_coords_tensor)
                    batch_point_labels.append(point_labels_tensor)
                
                logits = model(images, batch_point_coords, batch_point_labels)
                if logits.shape[2:] != masks.shape[1:]:
                    logits = torch.nn.functional.interpolate(
                        logits, 
                        size=masks.shape[1:], 
                        mode='bilinear', 
                        align_corners=False
                    )
                loss = criterion(logits, masks)
                val_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss': best_val_loss,
                'num_classes': NUM_CLASSES,
            }, 'best_multi_class_sam.pth')
            print(f'✅ Saved new best model with val loss: {best_val_loss:.4f}')
    
    return model

def generate_point_prompts(mask, num_points=5):
    h, w = mask.shape
    point_coords = []
    point_labels = []
    valid_mask = mask != IGNORE_CLASS_ID
    
    if not np.any(valid_mask):
        for _ in range(num_points):
            x = random.randint(0, w-1)
            y = random.randint(0, h-1)
            point_coords.append([x, y])
            point_labels.append(0)
        return np.array(point_coords), np.array(point_labels)
    
    valid_pixels = mask[valid_mask]
    unique_classes = np.unique(valid_pixels)
    unique_classes = unique_classes[unique_classes != 0]
    
    if len(unique_classes) == 0:
        valid_coords = np.argwhere(valid_mask)
        if len(valid_coords) > 0:
            sampled_indices = np.random.choice(len(valid_coords), min(num_points, len(valid_coords)), replace=False)
            for idx in sampled_indices:
                y, x = valid_coords[idx]
                point_coords.append([x, y])
                point_labels.append(0)
    else:
        points_per_class = max(1, num_points // len(unique_classes))
        for class_id in unique_classes:
            class_coords = np.argwhere((mask == class_id) & valid_mask)
            if len(class_coords) > 0:
                sampled_indices = np.random.choice(len(class_coords), min(points_per_class, len(class_coords)), replace=False)
                for idx in sampled_indices:
                    y, x = class_coords[idx]
                    point_coords.append([x, y])
                    point_labels.append(1)
        
        if len(point_coords) < num_points:
            background_coords = np.argwhere((mask == 0) & valid_mask)
            if len(background_coords) > 0:
                additional_points = num_points - len(point_coords)
                sampled_indices = np.random.choice(len(background_coords), min(additional_points, len(background_coords)), replace=False)
                for idx in sampled_indices:
                    y, x = background_coords[idx]
                    point_coords.append([x, y])
                    point_labels.append(0)
    
    if len(point_coords) < num_points:
        valid_coords = np.argwhere(valid_mask)
        if len(valid_coords) > 0:
            additional_points = num_points - len(point_coords)
            sampled_indices = np.random.choice(len(valid_coords), min(additional_points, len(valid_coords)), replace=False)
            for idx in sampled_indices:
                y, x = valid_coords[idx]
                point_coords.append([x, y])
                point_labels.append(0)
    
    return np.array(point_coords), np.array(point_labels)

# ==================== PREDICTION FUNCTION ====================
def predict_with_point_prompts(model, test_image_dir, output_dir, point_masks_dir, device='cuda'):
    os.makedirs(output_dir, exist_ok=True)
    model.eval()
    image_files = sorted([f for f in os.listdir(test_image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    for image_file in image_files:
        image_path = os.path.join(test_image_dir, image_file)
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not load image: {image_file}")
            continue
            
        original_image = image.copy()
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_size = image_rgb.shape[:2]
        
        input_image = cv2.resize(image_rgb, (1024, 1024))
        input_image = input_image.astype(np.float32) / 255.0
        input_image = (input_image - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
        
        input_image_tensor = torch.from_numpy(input_image).permute(2, 0, 1).float().to(device)
        
        point_mask_name = f'point_mask_{image_file}'
        point_mask_path = os.path.join(point_masks_dir, point_mask_name)
        if not os.path.exists(point_mask_path):
            point_mask_name = image_file
            point_mask_path = os.path.join(point_masks_dir, point_mask_name)
        if not os.path.exists(point_mask_path):
            print(f"Point mask not found: {point_mask_path}")
            continue
            
        point_mask = cv2.imread(point_mask_path, cv2.IMREAD_GRAYSCALE)
        if point_mask is None:
            print(f"Could not load point mask: {point_mask_path}")
            continue
        
        point_coords, point_labels = extract_points_alternative(point_mask)
        
        if len(point_coords) == 0:
            print(f"No valid points found, using default points")
            h, w = point_mask.shape
            for _ in range(5):
                x = random.randint(0, w-1)
                y = random.randint(0, h-1)
                point_coords.append([x, y])
                point_labels.append(1)
        
        # Convert to tensors with proper dimensions
        point_coords_tensor = torch.tensor(point_coords, dtype=torch.float32, device=device)
        point_labels_tensor = torch.tensor(point_labels, dtype=torch.float32, device=device)
        
        # Normalize coordinates
        point_coords_tensor = point_coords_tensor / torch.tensor([point_mask.shape[1], point_mask.shape[0]], device=device)
        
        with torch.no_grad():
            # Process single image
            logits = model.forward_single(input_image_tensor, point_coords_tensor, point_labels_tensor)
            logits = torch.nn.functional.interpolate(
                logits.unsqueeze(0), 
                size=original_size, 
                mode='bilinear', 
                align_corners=False
            )
            predictions = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        predictions_uint8 = predictions.astype(np.uint8)
        output_path = os.path.join(output_dir, f'mask_{image_file}')
        cv2.imwrite(output_path, predictions_uint8)
        
        print(f'Saved mask for {image_file} - Used {len(point_coords)} points')
    
    print("All predictions completed!")

# ==================== MAIN EXECUTION ====================
def main():
    print("Loading SAM model...")
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
    model = MultiClassSAMWrapper(sam, NUM_CLASSES)
    print(f"Model created with {NUM_CLASSES} classes")
    
    print("\nAnalyzing point masks...")
    analyze_point_masks(point_masks_dir)
    
    transform = SimpleTransform()
    dataset = MultiClassSAMDataset(train_image_dir, train_mask_dir, transform)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)
    
    print(f"Training on {len(train_dataset)} samples, validating on {len(val_dataset)} samples")
    
    print("Starting training...")
    trained_model = train_multi_class_sam(model, train_loader, val_loader, num_epochs=100, lr=1e-3)
    
    print("Starting prediction with 1-channel point masks...")
    predict_with_point_prompts(trained_model, test_image_dir, output_dir, point_masks_dir, device)
    
    print("Multi-class semantic segmentation completed!")

if __name__ == "__main__":
    main()

Number of classes: 19
Loading SAM model...
Model created with 19 classes

Analyzing point masks...
=== POINT MASK ANALYSIS ===
agric_1901.png: Shape: (256, 256), Unique values: [ 8 16 18]
  Contains ignore class (18): True
agric_1902.png: Shape: (256, 256), Unique values: [ 8 18]
  Contains ignore class (18): True
agric_1903.png: Shape: (256, 256), Unique values: [ 8 18]
  Contains ignore class (18): True
agric_1904.png: Shape: (256, 256), Unique values: [ 8 18]
  Contains ignore class (18): True
agric_1905.png: Shape: (256, 256), Unique values: [16 18]
  Contains ignore class (18): True
Total point masks found: 2100
Training on 504 samples, validating on 126 samples
Starting training...
Epoch 1, Batch 0, Loss: 2.7663
Epoch 1, Batch 5, Loss: 2.2924
Epoch 1, Batch 10, Loss: 2.0731
Epoch 1, Batch 15, Loss: 1.8070
Epoch 1, Batch 20, Loss: 1.5096
Epoch 1, Batch 25, Loss: 1.3752
Epoch 1, Batch 30, Loss: 1.1241
Epoch 1, Batch 35, Loss: 0.9032
Epoch 1, Batch 40, Loss: 0.7201
Epoch 1, Batch 45

In [7]:
import numpy as np
import os
from PIL import Image

# Define the class information
CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

def one_channel_to_rgb(one_channel_mask):
    """
    Convert 1-channel mask back to RGB for visualization.
    
    Args:
        one_channel_mask: numpy array of shape (H, W) with class indices
        
    Returns:
        rgb_mask: numpy array of shape (H, W, 3) with RGB values
    """
    h, w = one_channel_mask.shape
    rgb_mask = np.zeros((h, w, 3), dtype=np.uint8)
    
    for class_idx, class_info in CLASS_INFO.items():
        mask = one_channel_mask == class_idx
        rgb_mask[mask] = class_info["rgb"]
    
    return rgb_mask

def convert_1channel_to_rgb(input_folder, output_folder):
    """
    Convert all 1-channel masks in input_folder to RGB masks and save to output_folder.
    
    Args:
        input_folder: path to folder containing 1-channel masks
        output_folder: path to folder where RGB masks will be saved
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files in input folder
    supported_formats = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    image_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith(supported_formats)]
    
    print(f"Found {len(image_files)} masks to process in {input_folder}")
    
    # Process each image
    for i, filename in enumerate(image_files):
        # Load 1-channel mask
        input_path = os.path.join(input_folder, filename)
        one_channel_mask = np.array(Image.open(input_path))
        
        # Convert back to RGB
        rgb_mask = one_channel_to_rgb(one_channel_mask)
        
        # Save as PNG
        name, ext = os.path.splitext(filename)
        output_path = os.path.join(output_folder, f"{name}.png")
        
        # Save as PNG
        Image.fromarray(rgb_mask).save(output_path)
        
        if (i + 1) % 10 == 0 or (i + 1) == len(image_files):
            print(f"Processed {i+1}/{len(image_files)}: {filename} -> {output_path}")
    
    print(f"All masks converted and saved to {output_folder}")

# Set your paths here
input_folder = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output'
output_folder = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb'

# Check if input folder exists
if not os.path.exists(input_folder):
    print(f"Error: Input folder '{input_folder}' does not exist!")
else:
    # Convert all masks
    convert_1channel_to_rgb(input_folder, output_folder)
    print("Conversion completed successfully!")

Found 2100 masks to process in /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output
Processed 10/2100: mask_golfc_90.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_golfc_90.png
Processed 20/2100: mask_river_1306.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_river_1306.png
Processed 30/2100: mask_build_1698.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_build_1698.png
Processed 40/2100: mask_mobil_711.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_mobil_711.png
Processed 50/2100: mask_inter_2011.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_inter_2011.png
Processed 60/2100: mask_chapa_1719.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_chapa_1719.png
Processed 70/2100: mask_freew_325.png -> /home/

In [10]:
import torch
import numpy as np
import cv2
import os
from segment_anything import sam_model_registry

# ==================== CONFIGURATION ====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_image_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_images'
output_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output'
point_masks_dir = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_point_1cmasks'
sam_checkpoint = "/home/iiitdmk-param/Desktop/sam_vit_b.pth"
model_type = "vit_b"
fine_tuned_model_path = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/best_multi_class_sam.pth'

# Class information with ignore class
CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]},
    18: {"name": "point_mask", "rgb": [100, 100, 100]}  # This is the ignore class
}

IGNORE_CLASS_ID = 18  # Class ID for ignore
IGNORE_CLASS_RGB = [100, 100, 100]  # RGB value for ignore

NUM_CLASSES = len(CLASS_INFO)

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# ==================== MODEL DEFINITION ====================
class MultiClassSAMWrapper(torch.nn.Module):
    def __init__(self, sam_model, num_classes):
        super().__init__()
        self.sam = sam_model
        self.num_classes = num_classes
        
        # Multi-class head (same architecture as during training)
        self.multi_class_head = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, 3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout2d(0.1),
            torch.nn.Conv2d(64, 128, 3, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout2d(0.1),
            torch.nn.Conv2d(128, 64, 3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(64, num_classes, 1)
        )
        
        # Freeze SAM parameters
        for param in self.sam.parameters():
            param.requires_grad = False
    
    def forward_single(self, x, point_coords=None, point_labels=None):
        """Process single image with point prompts"""
        # Get image embeddings from SAM (frozen)
        with torch.no_grad():
            image_embeddings = self.sam.image_encoder(x.unsqueeze(0))
        
        if point_coords is not None and point_labels is not None:
            # Prepare points for SAM
            point_coords = point_coords.unsqueeze(0)  # [N, 2] -> [1, N, 2]
            point_labels = point_labels.unsqueeze(0)  # [N] -> [1, N]
            
            # Get prompt embeddings
            sparse_embeddings, dense_embeddings = self.sam.prompt_encoder(
                points=(point_coords, point_labels),
                boxes=None,
                masks=None,
            )
            
            # Use mask decoder with prompts
            low_res_masks, iou_predictions = self.sam.mask_decoder(
                image_embeddings=image_embeddings,
                image_pe=self.sam.prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=False,
            )
        else:
            # No points provided - use zero mask
            b, c, h, w = image_embeddings.shape
            low_res_masks = torch.zeros((b, 1, 256, 256), device=image_embeddings.device)
        
        # Pass through multi-class head
        logits = self.multi_class_head(low_res_masks)
        return logits.squeeze(0)  # Remove batch dimension

# ==================== POINT EXTRACTION ====================
def extract_points_from_point_mask(point_mask):
    """Extract point coordinates from your point annotation masks, ignoring class 18"""
    point_coords = []
    point_labels = []
    
    h, w = point_mask.shape
    
    # Find all non-zero AND non-ignore pixels (ignore class 18)
    valid_points = np.argwhere((point_mask > 0) & (point_mask != IGNORE_CLASS_ID))
    
    #print(f"Found {len(valid_points)} valid points (ignoring class {IGNORE_CLASS_ID})")
    
    for point in valid_points:
        y, x = point
        class_id = point_mask[y, x]
        
        # Convert to coordinates and labels
        point_coords.append([x, y])  # SAM expects [x, y] format
        point_labels.append(1)  # 1 = foreground point
        
        #print(f"  Point at ({x}, {y}) - class ID: {class_id}")
    
    # If no valid points found, use some default center points (but not from ignore regions)
    if len(point_coords) == 0:
        print("No valid points found in mask, using default points from non-ignore regions")
        
        # Find regions that are not ignore class
        non_ignore_mask = point_mask != IGNORE_CLASS_ID
        non_ignore_coords = np.argwhere(non_ignore_mask)
        
        if len(non_ignore_coords) > 0:
            # Sample 5 points from non-ignore regions
            num_points = min(5, len(non_ignore_coords))
            sampled_indices = np.random.choice(len(non_ignore_coords), num_points, replace=False)
            
            for idx in sampled_indices:
                y, x = non_ignore_coords[idx]
                point_coords.append([x, y])
                point_labels.append(1)
        else:
            # If everything is ignore class, use center points
            for i in range(5):
                x = int(w * (i + 1) / 6)
                y = int(h / 2)
                point_coords.append([x, y])
                point_labels.append(1)
    
    return np.array(point_coords), np.array(point_labels)

# ==================== IMAGE PREPROCESSING ====================
def preprocess_image(image):
    """Preprocess image for SAM"""
    # Resize to 1024x1024 (SAM's expected input size)
    input_image = cv2.resize(image, (1024, 1024))
    
    # Normalize like ImageNet
    input_image = input_image.astype(np.float32) / 255.0
    input_image = (input_image - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
    
    # Convert to tensor [C, H, W]
    input_image_tensor = torch.from_numpy(input_image).permute(2, 0, 1).float()
    
    return input_image_tensor

# ==================== MASK PROCESSING ====================
def process_mask_to_ignore_class(mask, ignore_rgb=IGNORE_CLASS_RGB):
    """Convert RGB mask to single-channel with proper ignore class handling"""
    if len(mask.shape) == 3:
        h, w, c = mask.shape
        processed_mask = np.zeros((h, w), dtype=np.uint8)
        ignore_pixels = np.all(mask == ignore_rgb, axis=-1)
        processed_mask[ignore_pixels] = IGNORE_CLASS_ID
        for class_id, info in CLASS_INFO.items():
            if class_id != IGNORE_CLASS_ID:  # Skip ignore class in mapping
                class_rgb = info["rgb"]
                class_pixels = np.all(mask == class_rgb, axis=-1)
                processed_mask[class_pixels] = class_id
    else:
        processed_mask = mask.copy()
    return processed_mask

# ==================== PREDICTION FUNCTION ====================
def predict_with_finetuned_model():
    """Load fine-tuned model and predict on test images with point guidance"""
    
    print("🚀 Starting prediction with fine-tuned SAM...")
    print(f"Ignore class ID: {IGNORE_CLASS_ID}, RGB: {IGNORE_CLASS_RGB}")
    
    # 1. Load original SAM model
    print("Loading SAM model...")
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
    
    # 2. Create wrapper with 19 classes
    model = MultiClassSAMWrapper(sam, num_classes=NUM_CLASSES)
    
    # 3. Load your fine-tuned weights
    print(f"Loading fine-tuned weights from: {fine_tuned_model_path}")
    checkpoint = torch.load(fine_tuned_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    print("✅ Model loaded successfully!")
    print(f"Number of classes: {NUM_CLASSES}")
    
    # 4. Get test images
    image_files = sorted([f for f in os.listdir(test_image_dir) 
                         if f.endswith(('.png', '.jpg', '.jpeg'))])
    
    print(f"📁 Found {len(image_files)} test images in {test_image_dir}")
    print(f"📁 Point masks directory: {point_masks_dir}")
    print(f"📁 Output directory: {output_dir}")
    
    # 5. Process each test image
    successful_predictions = 0
    
    for image_file in image_files:
        #print(f"\n{'='*60}")
        print(f"Processing: {image_file}")
        #print(f"{'='*60}")
        
        # Load test image
        image_path = os.path.join(test_image_dir, image_file)
        image = cv2.imread(image_path)
        if image is None:
            print(f"❌ Could not load image: {image_path}")
            continue
            
        original_image = image.copy()
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_size = image_rgb.shape[:2]  # (height, width)
        
        #print(f"Original image size: {original_size}")
        
        # Preprocess image
        input_image_tensor = preprocess_image(image_rgb).to(device)
        
        # Find corresponding point mask
        point_mask_name = image_file
        point_mask_path = os.path.join(point_masks_dir, point_mask_name)
        
        # Try different extensions if needed
        if not os.path.exists(point_mask_path):
            name_without_ext = os.path.splitext(image_file)[0]
            point_mask_path = os.path.join(point_masks_dir, name_without_ext + '.png')
        
        if not os.path.exists(point_mask_path):
            print(f"❌ Point mask not found: {point_mask_path}")
            continue
            
        point_mask = cv2.imread(point_mask_path, cv2.IMREAD_GRAYSCALE)
        if point_mask is None:
            print(f"❌ Could not load point mask: {point_mask_path}")
            continue
        
        #print(f"✅ Loaded point mask: {point_mask.shape}")
        #print(f"Point mask unique values: {np.unique(point_mask)}")
        
        # Extract points from your point annotations (ignoring class 18)
        point_coords, point_labels = extract_points_from_point_mask(point_mask)
        #print(f"🎯 Using {len(point_coords)} point prompts")
        
        # Convert to tensors and normalize coordinates to [0, 1] range
        point_coords_tensor = torch.tensor(point_coords, dtype=torch.float32, device=device)
        point_labels_tensor = torch.tensor(point_labels, dtype=torch.float32, device=device)
        
        # Normalize coordinates (SAM requirement)
        point_coords_tensor = point_coords_tensor / torch.tensor([point_mask.shape[1], point_mask.shape[0]], 
                                                               device=device)
        
        # Run inference with point guidance
        with torch.no_grad():
            #print("🔄 Running inference...")
            logits = model.forward_single(input_image_tensor, point_coords_tensor, point_labels_tensor)
            
            # Resize to original image size
            logits = torch.nn.functional.interpolate(
                logits.unsqueeze(0), 
                size=original_size, 
                mode='bilinear', 
                align_corners=False
            )
            
            # Get final prediction (class with highest probability)
            predictions = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        #print(f"Prediction shape: {predictions.shape}")
        unique_classes, counts = np.unique(predictions, return_counts=True)
        #print(f"Prediction class distribution:")
        for cls, count in zip(unique_classes, counts):
            class_name = CLASS_INFO.get(cls, {}).get("name", "unknown")
            #print(f"  Class {cls} ({class_name}): {count} pixels")
        
        # Save prediction
        predictions_uint8 = predictions.astype(np.uint8)
        output_filename = f'pred_{os.path.splitext(image_file)[0]}.png'
        output_path = os.path.join(output_dir, output_filename)
        cv2.imwrite(output_path, predictions_uint8)
        
        successful_predictions += 1
        print(f" Saved prediction: {output_path}")
    
    print(f"\n🎉 Prediction completed!")
    print(f"📊 Processed {successful_predictions}/{len(image_files)} images successfully")
    print(f"📁 All predictions saved in: {output_dir}")

# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    predict_with_finetuned_model()

🚀 Starting prediction with fine-tuned SAM...
Ignore class ID: 18, RGB: [100, 100, 100]
Loading SAM model...
Loading fine-tuned weights from: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/best_multi_class_sam.pth


/tmp/ipykernel_1843249/3888976404.py:207: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fine_tuned_model_path, map_location=device)


✅ Model loaded successfully!
Number of classes: 19
📁 Found 2100 test images in /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_images
📁 Point masks directory: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_point_1cmasks
📁 Output directory: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output

Processing: agric_1901.png
🔄 Running inference...
✅ Saved prediction: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output/pred_agric_1901.png

Processing: agric_1902.png
🔄 Running inference...
✅ Saved prediction: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output/pred_agric_1902.png

Processing: agric_1903.png
🔄 Running inference...
✅ Saved prediction: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output/pred_agric_1903.png

Processing: agric_1904.png
🔄 Running inference...
✅ Saved prediction: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output/pred_agric_1904.png

In [11]:
import numpy as np
import os
from PIL import Image

# Define the class information
CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}

def one_channel_to_rgb(one_channel_mask):
    """
    Convert 1-channel mask back to RGB for visualization.
    
    Args:
        one_channel_mask: numpy array of shape (H, W) with class indices
        
    Returns:
        rgb_mask: numpy array of shape (H, W, 3) with RGB values
    """
    h, w = one_channel_mask.shape
    rgb_mask = np.zeros((h, w, 3), dtype=np.uint8)
    
    for class_idx, class_info in CLASS_INFO.items():
        mask = one_channel_mask == class_idx
        rgb_mask[mask] = class_info["rgb"]
    
    return rgb_mask

def convert_1channel_to_rgb(input_folder, output_folder):
    """
    Convert all 1-channel masks in input_folder to RGB masks and save to output_folder.
    
    Args:
        input_folder: path to folder containing 1-channel masks
        output_folder: path to folder where RGB masks will be saved
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all image files in input folder
    supported_formats = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    image_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith(supported_formats)]
    
    print(f"Found {len(image_files)} masks to process in {input_folder}")
    
    # Process each image
    for i, filename in enumerate(image_files):
        # Load 1-channel mask
        input_path = os.path.join(input_folder, filename)
        one_channel_mask = np.array(Image.open(input_path))
        
        # Convert back to RGB
        rgb_mask = one_channel_to_rgb(one_channel_mask)
        
        # Save as PNG
        name, ext = os.path.splitext(filename)
        output_path = os.path.join(output_folder, f"{name}.png")
        
        # Save as PNG
        Image.fromarray(rgb_mask).save(output_path)
        
        if (i + 1) % 10 == 0 or (i + 1) == len(image_files):
            print(f"Processed {i+1}/{len(image_files)}: {filename} -> {output_path}")
    
    print(f"All masks converted and saved to {output_folder}")

# Set your paths here
input_folder = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output'
output_folder = '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb'

# Check if input folder exists
if not os.path.exists(input_folder):
    print(f"Error: Input folder '{input_folder}' does not exist!")
else:
    # Convert all masks
    convert_1channel_to_rgb(input_folder, output_folder)
    print("Conversion completed successfully!")

Found 4200 masks to process in /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output
Processed 10/4200: pred_parki_228.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/pred_parki_228.png
Processed 20/4200: mask_river_1386.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_river_1386.png
Processed 30/4200: pred_tenni_1092.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/pred_tenni_1092.png
Processed 40/4200: mask_fores_682.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_fores_682.png
Processed 50/4200: pred_river_1335.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/pred_river_1335.png
Processed 60/4200: mask_agric_1913.png -> /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/finetune_full_sam_output_rgb/mask_agric_1913.png
Processed 70/4200: pred_overp_1561.png -> /ho